<a href="https://colab.research.google.com/github/wsgcz/RoboCupVision-homework/blob/main/3_TensorIR_Tensor_Program_Abstraction_Case_Study_Action.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tensor Program Abstraction Case Study: TensorIR

## Install packages

For the purpose of this course, we will use some on-going development in tvm, which is an open source machine learning compilation framework. We provide the following command to install a packaged version for mlc course.

In [1]:
!python3 -m  pip install mlc-ai-nightly -f https://mlc.ai/wheels

Looking in links: https://mlc.ai/wheels
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.6/185.6 MB 6.4 MB/s eta 0:00:00


In [2]:
import tvm
from tvm.ir.module import IRModule
from tvm.script import tir as T
import numpy as np

In [38]:
a = np.arange(16).reshape(4,4)
b = np.arange(16,0,-1).reshape(4,4)

In [39]:
c_np = a + b
c_np

array([[16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16]])

In [41]:
b

array([[16, 15, 14, 13],
       [12, 11, 10,  9],
       [ 8,  7,  6,  5],
       [ 4,  3,  2,  1]])

In [43]:
# low-level numpy version
def lnumpy_add(a: np.ndarray, b: np.ndarray, c: np.ndarray):
  for i in range(4):
    for j in range(4):
      c[i, j] = a[i, j] + b[i, j]
c_lnumpy = np.empty((4, 4), dtype=np.int64)
lnumpy_add(a, b, c_lnumpy)
c_lnumpy


array([[16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16],
       [16, 16, 16, 16]])

In [46]:
# TensorIR version
@tvm.script.ir_module
class MyModule:
    @T.prim_func
    def add(a: T.Buffer[(4, 4), "int64"],
            b: T.Buffer[(4, 4), "int64"],
            c: T.Buffer[(4, 4), "int64"]):
      T.func_attr({"global_symbol": "add", "tir.noalias": True})
      for i, j in T.grid(4,4):
        with T.block("C"):
          vi = T.axis.spatial(4, i)
          vj = T.axis.spatial(4, j)
          c[vi, vj] = a[vi, vj] + b[vi, vj]

rt_lib = tvm.build(MyModule, target="llvm")
a_tvm = tvm.nd.array(a)
b_tvm = tvm.nd.array(b)
c_tvm = tvm.nd.array(np.empty((4, 4), dtype = np.int64))
rt_lib["add"](a_tvm, b_tvm, c_tvm)
np.testing.assert_allclose(c_tvm.numpy(), c_np, rtol=1e-5)

/tmp/ipython-input-4274048251.py:5: DeprecationWarning: T.Buffer[...] is deprecated, use T.Buffer(...) instead
  def add(a: T.Buffer[(4, 4), "int64"],
/tmp/ipython-input-4274048251.py:6: DeprecationWarning: T.Buffer[...] is deprecated, use T.Buffer(...) instead
  b: T.Buffer[(4, 4), "int64"],
/tmp/ipython-input-4274048251.py:7: DeprecationWarning: T.Buffer[...] is deprecated, use T.Buffer(...) instead
  c: T.Buffer[(4, 4), "int64"]):
<ast>:3: DeprecationWarning: T.Buffer[...] is deprecated, use T.Buffer(...) instead
<ast>:4: DeprecationWarning: T.Buffer[...] is deprecated, use T.Buffer(...) instead


In [ ]:
dtype = "float32"
a_np = np.random.rand(128, 128).astype(dtype)
b_np = np.random.rand(128, 128).astype(dtype)
# a @ b is equivalent to np.matmul(a, b)
c_mm_relu = np.maximum(a_np @ b_np, 0)

In [47]:
a = np.arange(16).reshape(4,4)
b = np.arange(4,0,-1).reshape(4)

In [48]:
c_np = a + b
c_np

array([[ 4,  4,  4,  4],
       [ 8,  8,  8,  8],
       [12, 12, 12, 12],
       [16, 16, 16, 16]])

In [49]:
a,b

(array([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11],
        [12, 13, 14, 15]]),
 array([4, 3, 2, 1]))

In [17]:
import IPython

In [21]:
# init data
a = np.arange(16).reshape(4, 4)
b = np.arange(4, 0, -1).reshape(4)
# numpy version
c_np = a + b
c_np

array([[ 4,  4,  4,  4],
       [ 8,  8,  8,  8],
       [12, 12, 12, 12],
       [16, 16, 16, 16]])

In [22]:
@tvm.script.ir_module
class MyAdd:
  @T.prim_func
  def add(A: T.Buffer((4,4), "int64"),
          B: T.Buffer((4), "int64"),
          C: T.Buffer((4,4), "int64")):
    T.func_attr({"global_symbol": "add", "tir.noalias": True})
    for i, j in T.grid(4,4):
      with T.block("C"):
        vi = T.axis.spatial(4,i)
        vj = T.axis.spatial(4,j)
        C[vi, vj] = A[vi, vj] + B[vj]

rt_lib = tvm.build(MyAdd, target="llvm")
a_tvm = tvm.nd.array(a)
b_tvm = tvm.nd.array(b)
c_tvm = tvm.nd.array(np.empty((4, 4), dtype=np.int64))
rt_lib["add"](a_tvm, b_tvm, c_tvm)
np.testing.assert_allclose(c_tvm.numpy(), c_np, rtol=1e-5)


In [3]:
N, CI, H, W, CO, K = 1, 1, 8, 8, 2, 3
OUT_H, OUT_W = H - K + 1, W - K + 1
data = np.arange(N*CI*H*W).reshape(N, CI, H, W)
weight = np.arange(CO*CI*K*K).reshape(CO, CI, K, K)

In [4]:
import torch

data_torch = torch.Tensor(data)
weight_torch = torch.Tensor(weight)
conv_torch = torch.nn.functional.conv2d(data_torch, weight_torch)
conv_torch = conv_torch.numpy().astype(np.int64)
conv_torch

array([[[[ 474,  510,  546,  582,  618,  654],
         [ 762,  798,  834,  870,  906,  942],
         [1050, 1086, 1122, 1158, 1194, 1230],
         [1338, 1374, 1410, 1446, 1482, 1518],
         [1626, 1662, 1698, 1734, 1770, 1806],
         [1914, 1950, 1986, 2022, 2058, 2094]],

        [[1203, 1320, 1437, 1554, 1671, 1788],
         [2139, 2256, 2373, 2490, 2607, 2724],
         [3075, 3192, 3309, 3426, 3543, 3660],
         [4011, 4128, 4245, 4362, 4479, 4596],
         [4947, 5064, 5181, 5298, 5415, 5532],
         [5883, 6000, 6117, 6234, 6351, 6468]]]])

In [12]:
@tvm.script.ir_module
class MyConv:
  @T.prim_func
  def conv(data : T.Buffer((N, CI, H, W), dtype="int64"),
           weight : T.Buffer((CO, CI, K, K), dtype="int64"),
           conv : T.Buffer((N, CO, H-K+1, W-K+1), dtype="int64")):
    T.func_attr({"global_symbol": "conv", "tir.noalias": True})
    for i,j,k,l,m,n,o in T.grid(N, CO, H-K+1, W-K+1, CI, K, K):
      with T.block("conv"):
        vi = T.axis.spatial(N,i)
        vj = T.axis.spatial(CO,j)
        vk = T.axis.spatial(H-K+1,k)
        vl = T.axis.spatial(W-K+1,l)
        vm = T.axis.reduce(CI,m)
        vn = T.axis.reduce(K,n)
        vo = T.axis.reduce(K,o)
        with T.init():
          conv[vi,vj,vk,vl] = T.int64(0)
        conv[vi,vj,vk,vl] = conv[vi,vj,vk,vl] + data[vi, vm, vk+vn, vl+vo] * weight[vj, vm, vn, vo]


rt_lib = tvm.build(MyConv, target="llvm")
data_tvm = tvm.nd.array(data)
weight_tvm = tvm.nd.array(weight)
conv_tvm = tvm.nd.array(np.empty((N, CO, OUT_H, OUT_W), dtype=np.int64))
rt_lib["conv"](data_tvm, weight_tvm, conv_tvm)
np.testing.assert_allclose(conv_tvm.numpy(), conv_torch, rtol=1e-5)


In [13]:
conv_tvm.numpy().shape

(1, 2, 6, 6)

In [14]:
conv_torch.shape

(1, 2, 6, 6)

In [23]:
@tvm.script.ir_module
class MyAdd:
  @T.prim_func
  def add(A: T.Buffer((4, 4), "int64"),
          B: T.Buffer((4, 4), "int64"),
          C: T.Buffer((4, 4), "int64")):
    T.func_attr({"global_symbol": "add"})
    for i, j in T.grid(4, 4):
      with T.block("C"):
        vi = T.axis.spatial(4, i)
        vj = T.axis.spatial(4, j)
        C[vi, vj] = A[vi, vj] + B[vi, vj]

sch = tvm.tir.Schedule(MyAdd)
block = sch.get_block("C", func_name="add")
i, j = sch.get_loops(block)
i0, i1 = sch.split(i, factors=[2, 2])
sch.parallel(i0)
sch.unroll(i1)
sch.vectorize(j)
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def add(A: T.Buffer((4, 4), "int64"), B: T.Buffer((4, 4), "int64"), C: T.Buffer((4, 4), "int64")):
        # with T.block("root"):
        for i_0 in T.parallel(2):
            for i_1 in T.unroll(2):
                for j in T.vectorized(4):
                    with T.block("C"):
                        vi = T.axis.spatial(4, i_0 * 2 + i_1)
                        vj = T.axis.spatial(4, j)
                        T.reads(A[vi, vj], B[vi, vj])
                        T.writes(C[vi, vj])
                        C[vi, vj] = A[vi, vj] + B[vi, vj]

In [ ]:
# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def add(A: T.Buffer((4, 4), "int64"), B: T.Buffer((4, 4), "int64"), C: T.Buffer((4, 4), "int64")):
        # with T.block("root"):
        for i_0 in T.parallel(2):
            for i_1 in T.unroll(2):
                for j in T.vectorized(4):
                    with T.block("C"):
                        vi = T.axis.spatial(4, i_0 * 2 + i_1)
                        vj = T.axis.spatial(4, j)
                        T.reads(A[vi, vj], B[vi, vj])
                        T.writes(C[vi, vj])

In [24]:
def lnumpy_mm_relu_v2(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    Y = np.empty((16, 128, 128), dtype="float32")
    for n in range(16):
        for i in range(128):
            for j in range(128):
                for k in range(128):
                    if k == 0:
                        Y[n, i, j] = 0
                    Y[n, i, j] = Y[n, i, j] + A[n, i, k] * B[n, k, j]
    for n in range(16):
        for i in range(128):
            for j in range(128):
                C[n, i, j] = max(Y[n, i, j], 0)

In [30]:
dtype = "float32"
a_np = np.random.rand(16,128, 128).astype(dtype)
b_np = np.random.rand(16,128, 128).astype(dtype)
# a @ b is equivalent to np.matmul(a, b)
c_mm_relu = np.maximum(a_np @ b_np, 0)


In [62]:
@tvm.script.ir_module
class MyBmmRelu:
  @T.prim_func
  def bmm_relu(A: T.Buffer((16, 128, 128), "float32"),
               B: T.Buffer((16, 128, 128), "float32"),
               C: T.Buffer((16, 128, 128), "float32")):
    T.func_attr({"global_symbol": "bmm_relu", "tir.noalias": True})
    Y = T.alloc_buffer((16,128,128), dtype="float32")
    for n,i,j,k in T.grid(16,128,128,128):
      with T.block("C"):
        vn, vi, vj, vk = T.axis.remap("SSSR", [n,i,j,k])
        with T.init():
          Y[vn, vi, vj] = T.float32(0)
        Y[vn, vi, vj] = Y[vn, vi, vj] + A[vn, vi, vk] * B[vn, vk, vj]
    for n,i,j in T.grid(16,128,128):
      with T.block("D"):
        vn, vi, vj = T.axis.remap("SSS", [n,i,j])
        C[vn, vi, vj] = T.max(Y[vn, vi, vj],T.float32(0))

sch = tvm.tir.Schedule(MyBmmRelu)
IPython.display.Code(sch.mod.script(), language="python")
# Also please validate your result


# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"), B: T.Buffer((16, 128, 128), "float32"), C: T.Buffer((16, 128, 128), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        Y = T.alloc_buffer((16, 128, 128))
        for n, i, j, k in T.grid(16, 128, 128, 128):
            with T.block("C"):
                vn, vi, vj, vk = T.axis.remap("SSSR", [n, i, j, k])
                T.reads(A[vn, vi, vk], B[vn, vk, vj])
                T.writes(Y[vn, vi, vj])
                with T.init():
                    Y[vn, vi, vj] = T.float32(0.0)
                Y[vn, vi, vj] = Y[vn, vi, vj] + A[vn, vi, vk] * B[vn, vk, vj]
        for n, i, j in T.grid(16, 128, 128):
            with T.block("D"):
                vn, vi, vj = T.axis.remap("SSS", [n, i, j])
                T.reads(Y[vn, vi, vj])
                T.writes(C[vn, vi, vj])
                C[vn, vi, vj] = T.max(Y[vn, vi, vj], T.float32(0.0))

In [63]:
rt_lib = tvm.build(MyBmmRelu, target="llvm")
a_tvm = tvm.nd.array(a_np)
b_tvm = tvm.nd.array(b_np)
c_tvm = tvm.nd.array(np.empty((16,128,128), dtype=np.float32))
rt_lib["bmm_relu"](a_tvm, b_tvm, c_tvm)
np.testing.assert_allclose(c_tvm.numpy(), c_mm_relu, rtol=1e-5)

In [71]:
sch = tvm.tir.Schedule(MyBmmRelu)
# # TODO: transformations
# # Hints: you can use
# # `IPython.display.Code(sch.mod.script(), language="python")`
# or `print(sch.mod.script())`
# to show the current program at any time during the transformation.

# sch = tvm.tir.Schedule(MyAdd)
# block = sch.get_block("C", func_name="add")
# i, j = sch.get_loops(block)
# i0, i1 = sch.split(i, factors=[2, 2])
# sch.parallel(i0)
# sch.unroll(i1)
# sch.vectorize(j)
# IPython.display.Code(sch.mod.script(), language="python")
# Step 1. Get blocks
block_C = sch.get_block("C", func_name="bmm_relu")

# Step 2. Get loops
n, i, j, k = sch.get_loops(block_C)
sch.parallel(n)

# Step 3. Organize the loops
j0, j1 = sch.split(j, factors=[16,8])
sch.vectorize(j1)
#sch.reorder(n,i,j0,k0,k1,j1)
sch.reorder(n,i,j0,k,j1)
block_D = sch.get_block("D", func_name="bmm_relu")

# Step 2. Get loops
sch.reverse_compute_at(block_D,j0)

# Step 4. decompose reduction
sch.decompose_reduction(block_C, k)
k0, k1 = sch.split(k, factors=[32,4])

# Step 5. vectorize / parallel / unroll
sch.unroll(k1)
IPython.display.Code(sch.mod.script(), language="python")

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"), B: T.Buffer((16, 128, 128), "float32"), C: T.Buffer((16, 128, 128), "float32")):
        T.func_attr({"tir.noalias": T.bool(True)})
        # with T.block("root"):
        Y = T.alloc_buffer((16, 128, 128))
        for n in T.parallel(16):
            for i, j_0 in T.grid(128, 16):
                for j_1_init in T.vectorized(8):
                    with T.block("C_init"):
                        vn, vi = T.axis.remap("SS", [n, i])
                        vj = T.axis.spatial(128, j_0 * 8 + j_1_init)
                        T.reads()
                        T.writes(Y[vn, vi, vj])
                        Y[vn, vi, vj] = T.float32(0.0)
                for k_0 in range(32):
                    for k_1 in T.unroll(4):
                        for j_1 in T.vectorized(8):
                            with T.block("C_update"):
                                vn, vi = T.axis.remap("SS", [n, i])
                                vj = T.axis.spatial(128, j_0 * 8 + j_1)
                                vk = T.axis.reduce(128, k_0 * 4 + k_1)
                                T.reads(Y[vn, vi, vj], A[vn, vi, vk], B[vn, vk, vj])
                                T.writes(Y[vn, vi, vj])
                                Y[vn, vi, vj] = Y[vn, vi, vj] + A[vn, vi, vk] * B[vn, vk, vj]
                for ax0 in range(8):
                    with T.block("D"):
                        vn, vi = T.axis.remap("SS", [n, i])
                        vj = T.axis.spatial(128, j_0 * 8 + ax0)
                        T.reads(Y[vn, vi, vj])
                        T.writes(C[vn, vi, vj])
                        C[vn, vi, vj] = T.max(Y[vn, vi, vj], T.float32(0.0))

In [72]:
@tvm.script.ir_module
class TargetModule:
    @T.prim_func
    def bmm_relu(A: T.Buffer((16, 128, 128), "float32"), B: T.Buffer((16, 128, 128), "float32"), C: T.Buffer((16, 128, 128), "float32")) -> None:
        T.func_attr({"global_symbol": "bmm_relu", "tir.noalias": True})
        Y = T.alloc_buffer([16, 128, 128], dtype="float32")
        for i0 in T.parallel(16): # i0 => n
            for i1, i2_0 in T.grid(128, 16): # i1 => i, i2_0 =>
                for ax0_init in T.vectorized(8):
                    with T.block("Y_init"):
                        n, i = T.axis.remap("SS", [i0, i1])
                        j = T.axis.spatial(128, i2_0 * 8 + ax0_init)
                        Y[n, i, j] = T.float32(0)
                for ax1_0 in T.serial(32):
                    for ax1_1 in T.unroll(4):
                        for ax0 in T.serial(8):
                            with T.block("Y_update"):
                                n, i = T.axis.remap("SS", [i0, i1])
                                j = T.axis.spatial(128, i2_0 * 8 + ax0)
                                k = T.axis.reduce(128, ax1_0 * 4 + ax1_1)
                                Y[n, i, j] = Y[n, i, j] + A[n, i, k] * B[n, k, j]
                for i2_1 in T.vectorized(8):
                    with T.block("C"):
                        n, i = T.axis.remap("SS", [i0, i1])
                        j = T.axis.spatial(128, i2_0 * 8 + i2_1)
                        C[n, i, j] = T.max(Y[n, i, j], T.float32(0))

In [74]:
print("Pass")

Pass
